# Du, Lai, Hu & Ye (2026) on MC_RTT (DANDI:000129)

Paper doi:10.64898/2026.08.04.742876 (claims file written from the bioRxiv full text, without the
supplement). Dataset: DANDI:000129 (MC_RTT, Neural Latents Benchmark), pinned at version
**0.241017.1444**, the version the paper cites (Methods 4.2, Data availability) and the only published
version; file `sub-Indy/sub-Indy_desc-train_behavior+ecephys.nwb`, streamed. See
`02_link_development.ipynb` for the full claims table.

| Claim | Status |
|---|---|
| rtt-data | tested |
| rtt-gain | tested |
| rff-rtt | tested |
| estimator-family-nlb | MC_RTT part tested (reading (a) of the nonlinear context); MC_Maze part not run (DANDI:000139 is not one of this attempt's datasets) |
| maze-gain, rff-maze | blocked (DANDI:000139 not used) |

In [ ]:
import collections, resource, time, zlib
import numpy as np
import pandas as pd
import du2026 as d

PAPER = "du-2026"
def claim_seed(claim_id):
    return zlib.crc32(f"{PAPER}/{claim_id}".encode())
T0 = time.time()
EXTRA = (1000, 2000, 3000, 4000)

rtt_assets = d.list_assets(d.RTT_ID, d.RTT_VERSION)
print([(a["path"], a["size"]) for a in rtt_assets])
train_asset = next(a for a in rtt_assets if a["path"] == d.RTT_TRAIN_PATH)
raw = d.load_rtt_raw(train_asset["asset_id"])
ds = d.build_rtt_dataset(raw)

In [ ]:
print("behaviour series:", raw.descriptions)
print(f"behaviour sampling: starting time {raw.behavior_timestamps_head}, rate {raw.behavior_rate} Hz; 20 ms bins: {raw.n_bins}")
print(f"units: {raw.n_units}; spike counts per 20 ms bin range {raw.spike_counts.min():.0f} to {raw.spike_counts.max():.0f}, mean rate {raw.spike_counts.mean() / 0.02:.1f} Hz")
print("trials table:", raw.trials.shape, "split counts", raw.trials.split.value_counts().to_dict())
dur = (raw.trials.stop_time - raw.trials.start_time)
print(f"block durations {dur.min():.3f} to {dur.max():.3f} s; first start {raw.trials.start_time.min()}, last stop {raw.trials.stop_time.max()} s; record {raw.n_bins * 0.02:.2f} s")
print(f"finger velocity range {np.nanmin(raw.finger_vel):.1f} to {np.nanmax(raw.finger_vel):.1f}; cursor {np.nanmin(raw.cursor_pos):.1f} to {np.nanmax(raw.cursor_pos):.1f}; target {np.nanmin(raw.target_pos):.1f} to {np.nanmax(raw.target_pos):.1f}")
bad = ~np.isfinite(raw.finger_vel).all(axis=1) | ~np.isfinite(raw.cursor_pos).all(axis=1) | ~np.isfinite(raw.target_pos).all(axis=1)
print(f"20 ms bins with a missing (NaN) behaviour value: {int(bad.sum())} of {raw.n_bins}; NaN in the analysis arrays: "
      f"behaviour {int(np.isnan(ds.behavior).sum())}, context {int(np.isnan(ds.context).sum())}")
print("analysis arrays: neural", ds.neural.shape, "context", ds.context.shape, "behaviour", ds.behavior.shape, "train/val", len(ds.train_indices), len(ds.test_indices))
# velocity against the derivative of cursor position, as a check of units and alignment (mm/s against mm per 20 ms bin)
dcur = np.gradient(raw.cursor_pos, axis=0) / 0.02
for lag in (-5, -2, 0, 2, 5):
    a, b = (raw.finger_vel[max(0, lag):len(dcur) + min(0, lag)], dcur[max(0, -lag):len(dcur) - max(0, lag)])
    ok = np.isfinite(a).all(axis=1) & np.isfinite(b).all(axis=1)
    print(f"corr(finger velocity, cursor velocity shifted {lag} bins): x {np.corrcoef(a[ok, 0], b[ok, 0])[0, 1]:.3f}, y {np.corrcoef(a[ok, 1], b[ok, 1])[0, 1]:.3f}")
# clock check: decode finger velocity from 100 ms of spikes ending `lag` bins before the behaviour bin
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
bins_ = np.arange(50, raw.n_bins - 50)
bins_ = bins_[~bad[bins_]]
row = {}
for lag in (-25, -10, -4, 0, 2, 4, 6, 10, 25):
    X = np.stack([raw.spike_counts[b - lag - 5:b - lag].reshape(-1) for b in bins_])
    half = len(bins_) * 3 // 4
    m = make_pipeline(StandardScaler(), Ridge(alpha=1000.0)).fit(X[:half], raw.finger_vel[bins_[:half]])
    row[lag] = round(r2_score(raw.finger_vel[bins_[half:]], m.predict(X[half:]), multioutput="variance_weighted"), 3)
print("held-out R2 (last quarter of the record) by lag in bins:", row)
print(f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## rtt-data

> The MC_RTT training file holds 1,080 fixed analysis blocks, 130 units, finger velocity, cursor
> position and target position, with a release split of 810 train and 270 validation blocks.

In [ ]:
print(f"blocks kept by the mc_rtt.py rule: {len(ds.neural)} of {len(raw.trials)} (expected 1080); units {raw.n_units} (expected 130); "
      f"train {len(ds.train_indices)} (810), validation {len(ds.test_indices)} (270); bins per block {ds.neural.shape[1]}")
print("series present: finger_vel, cursor_pos, target_pos:", sorted(raw.descriptions))

**Result: reproduced.** The training file's trials table has 1,080 blocks of 600 ms, all kept by the loader's rule (21 bins each after the warm-up), with 130 units and the finger velocity, cursor position and target position series; the split labels give 810 train and 270 validation blocks.

## rtt-gain

> In MC_RTT, target plus lag-matched cursor-position history gave context-only R2 of 0.6031, and fully
> nested neural gain beyond it was 0.06846 (conditional interval 0.05371 to 0.08478), context RMSE
> falling from 39.78 to 36.19 (a 17.25% MSE reduction), with component gains 0.0836 and 0.0513 and
> replacement gain -0.02297.

Estimator seed 20260726 (`run_primary_analyses.py`); the release split is fixed, and the estimator's
folds and inner splits are also run under seeds plus 1000 to 4000. The same fits give the three
nonlinear-neural maps (rff-rtt).

In [ ]:
SEED = 20260726
jobs = [(f"nested_{SEED + k}", {}, "nested", dict(seed=SEED + k, rff_seeds=d.RFF_SEEDS if k == 0 else ()), False) for k in (0,) + EXTRA]
jobs_family = [("block", {}, "block", {"seed": SEED}, False), ("rffctx", {}, "rff_context", {"seed": SEED}, False)]
t = time.time()
out = d.run_parallel(d.rtt_jobs, [(train_asset["asset_id"], [j]) for j in jobs + jobs_family])
res = {k: v for o in out for k, v in o.items()}
print(f"fits in {(time.time() - t) / 60:.1f} min")
r = res[f"nested_{SEED}"]
exp = [("context-only R2", r["context_r2"], 0.6031, 0.06031), ("fully nested gain", r["gain"], 0.06846, 0.01632),
       ("context RMSE", r["context_rmse"], 39.78, 3.978), ("context-plus-neural RMSE", r["exact_rmse"], 36.19, 3.619),
       ("% MSE reduction", r["mse_reduction_percent"], 17.25, 1.725), ("gain, component 1", r["component_gain"][0], 0.0836, 0.00836),
       ("gain, component 2", r["component_gain"][1], 0.0513, 0.00513), ("replacement gain", r["replacement_gain"], -0.02297, 0.002297)]
for label, v, e, tol in exp:
    print(f"{label}: {v:.5f} (expected {e} +- {tol}; within: {abs(v - e) <= tol})")
for k in (0,) + EXTRA:
    rr = res[f"nested_{SEED + k}"]
    print(f"seed {SEED + k}: gain {rr['gain']:.5f}, replacement {rr['replacement_gain']:.5f}, context R2 {rr['context_r2']:.4f}")
print("penalties (context, residualizer, residual learner):", r["context_alpha"], r["residualizer_alpha"], r["innovation_alpha"])

**Result: reproduced.** Context-only R2 is 0.60308, the fully nested gain 0.06846, the context RMSE 39.781 and the context-plus-neural RMSE 36.188 (a 17.248% MSE reduction), the component gains 0.08359 and 0.05125, and the replacement gain -0.02297; all eight equal the paper's values at its precision. Under four other estimator seeds the gain is 0.06821 to 0.06855 and the replacement gain -0.02300 to -0.02332.

## rff-rtt

> Under three prespecified nonlinear-neural random-feature maps, MC_RTT gain stayed positive (0.01491 to
> 0.01685).

In [ ]:
g = {s: res[f"nested_{SEED}"]["rff"][s]["gain"] for s in d.RFF_SEEDS}
print({s: round(v, 5) for s, v in g.items()}, "penalties", {s: res[f"nested_{SEED}"]["rff"][s]["alpha"] for s in d.RFF_SEEDS})
lo, hi = 0.01491 - 0.001491, 0.01685 + 0.001685
print(f"all positive: {all(v > 0 for v in g.values())}; all inside {lo:.5f}..{hi:.5f}: {all(lo <= v <= hi for v in g.values())}")
print("replacement gains under the maps:", {s: round(res[f"nested_{SEED}"]["rff"][s]["replacement_gain"], 5) for s in d.RFF_SEEDS})

**Result: reproduced.** The three nonlinear-neural maps give MC_RTT gains of 0.01685, 0.01501 and 0.01491, all positive and spanning the paper's range of 0.01491 to 0.01685.

## estimator-family-nlb

> Under the separate-block and nonlinear-context estimators, MC_RTT gain remained positive while
> MC_Maze remained near zero.

In [ ]:
print(f"block penalties: gain {res['block']['gain']:.5f} (penalties {res['block']['penalties']}); "
      f"nonlinear context (D 256, g chosen {res['rffctx']['gamma']}): gain {res['rffctx']['gain']:.5f}; fully nested linear {res[f'nested_{SEED}']['gain']:.5f}")
print("MC_RTT positive under both families:", res["block"]["gain"] > 0 and res["rffctx"]["gain"] > 0)
print(f"run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

**Result: reproduced (MC_RTT part).** The block-penalty estimator gives an MC_RTT gain of 0.0690 and the nonlinear-context estimator 0.04619, both positive, as the criterion requires. The MC_Maze part of the statement was not tested: DANDI:000139 was not read in this attempt.

## Summary

| Claim | Outcome |
|---|---|
| rtt-data | reproduced |
| rtt-gain | reproduced |
| rff-rtt | reproduced |
| estimator-family-nlb | reproduced (MC_RTT part only) |
| maze-gain, rff-maze | blocked (MC_Maze, DANDI:000139, not read in this attempt) |

What was found about the dataset: the counts are as the paper states. The behaviour series hold 15
missing (NaN) 20 ms bins, none inside the analysis blocks. The units of the behaviour series are
consistent once the `conversion` attribute is read: `unit` m/s or meters with conversion 0.001, so
the stored values are mm/s and mm, as the descriptions say; the paper's statement that the NLB unit
fields and descriptions disagree reads the unit without the conversion. Finger velocity follows the
derivative of cursor position at lag 0 (correlation 0.993 and 0.996), and velocity decoding from
spikes peaks 2 bins (40 ms) after the spikes, so the spikes and behaviour share a clock.